# Pipeline Integrado de Tratamento e Sanitização de Séries Históricas (B3)
**Autor:** Pedro Reis  
**Escopo:** Consolidação de Dados para Análise de Portfólios (MPT, PMPT, BL)

Este notebook implementa, em um único fluxo reproduzível, todas as etapas de tratamento dos dados de cotações de fechamento extraídos da Economatica. Substitui e unifica os dois notebooks anteriores (`1_tratamento_planilhas_economatica` e `2_Sanitizacao_Filtro_Liquidez_95`), corrigindo as falhas metodológicas identificadas na revisão.

### Pipeline Consolidado:
1. **Ingestão dos relatórios Economatica:** leitura defensiva dos arquivos `.xlsx`, com filtro de lockfiles do Excel e seleção de colunas por nome.
2. **Concatenação e alinhamento cronológico:** união lateral por índice de data com ordenação explícita.
3. **Filtro temporal:** janela amostral estrita entre 01/01/2010 e 31/12/2025.
4. **Remoção de feriados nacionais:** purgação de linhas onde a B3 não operou (a Economatica entrega esses dias como linhas com `-`).
5. **Filtro de liquidez:** retenção de ativos com presença em pregão $\geq 95\%$ sobre os dias efetivos de mercado.
6. **Filtro de integridade inicial:** exclusão de ativos cujo primeiro pregão real é posterior ao início da janela amostral, evitando lookahead bias por imputação retroativa.
7. **Imputação residual:** *Forward Fill* exclusivo (sem *Backward Fill*) sobre lacunas legítimas (suspensões, leilões intradiários).
8. **Testes de integridade:** verificação de unicidade do índice, monotonicidade cronológica e positividade dos preços.
9. **Persistência auditável:** gravação da matriz final e do log dos ativos descartados, com justificativa de exclusão.

## Configuração do Ambiente

Definição de caminhos relativos ancorados na raiz do projeto, configuração do sistema de logs e criação automatizada dos diretórios de saída. O uso de `pathlib.Path` com caminhos relativos garante portabilidade entre máquinas e independência do diretório de trabalho do usuário.

In [19]:
from pathlib import Path
import logging
import pandas as pd
import os

# Configuração do sistema de logs
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] - %(message)s"
)

# Caminhos relativos à raiz do projeto (usando pathlib puramente)
parent_path = Path.cwd().parent.parent

# Substitua o os.path.join por operadores de barra (/) do pathlib
DIRETORIO_ORIGEM = parent_path / "data" / "dados_economatica"
DIRETORIO_DESTINO = parent_path / "data" / "Matriz_Precos"

# Agora você pode descomentar e usar o método do pathlib com segurança:
DIRETORIO_DESTINO.mkdir(parents=True, exist_ok=True)

# Criação automatizada do diretório de destino
#DIRETORIO_DESTINO.mkdir(parents=True, exist_ok=True)

# Parâmetros metodológicos do pipeline (concentrados para auditoria)
DATA_INICIO = pd.to_datetime('2010-01-01')
DATA_FIM    = pd.to_datetime('2025-12-31')
LIMIAR_PRESENCA = 0.95   # Critério de presença em pregão (alinhado ao IBOVESPA)

print(f"[OK] Diretório de origem:  {DIRETORIO_ORIGEM}")
print(f"[OK] Diretório de destino: {DIRETORIO_DESTINO}")
print(f"[OK] Janela temporal: {DATA_INICIO.date()} a {DATA_FIM.date()}")
print(f"[OK] Limiar de presença:  {LIMIAR_PRESENCA:.0%}")

[OK] Diretório de origem:  c:\VSCodeWorkspace\TCC_Final\data\dados_economatica
[OK] Diretório de destino: c:\VSCodeWorkspace\TCC_Final\data\Matriz_Precos
[OK] Janela temporal: 2010-01-01 a 2025-12-31
[OK] Limiar de presença:  95%


## Etapa I — Ingestão dos Relatórios Economatica

Varredura do diretório de origem buscando arquivos `.xlsx`, com duas correções de robustez em relação ao pipeline anterior:

1. **Filtro de *lockfiles*:** o Excel cria arquivos temporários com prefixo `~$` enquanto a planilha está aberta. Esses arquivos não são *workbooks* válidos e produzem falsos erros se forem capturados pelo `glob`. O filtro `not nome.startswith('~$')` elimina o problema na origem.
2. **Seleção por nome de coluna:** em lugar do índice posicional `usecols=[0, 4]`, a leitura é parametrizada por nome (`usecols=['Data', 'Fechamento']`). Caso a Economatica altere a ordem das colunas em exportações futuras, o pipeline falha de forma explícita em vez de ler silenciosamente a coluna errada.

Os parâmetros de leitura permanecem:
- `skiprows=3` descarta o cabeçalho textual da Economatica (nome do ativo, moeda, linha em branco).
- `na_values=['-']` converte o marcador de feriado/sem-negócio da Economatica em `NaN`.

In [20]:
# Mapeamento dos arquivos Excel, com exclusão de lockfiles temporários do Office
arquivos_excel = [
    p for p in DIRETORIO_ORIGEM.glob("*.xlsx")
    if not p.name.startswith("~$")
]
print(f"[+] Relatórios identificados (excluindo lockfiles): {len(arquivos_excel)}")

# Lista para armazenamento dos DataFrames individuais preparados
lista_dataframes = []
falhas_leitura = []

for caminho_arquivo in arquivos_excel:
    nome_arquivo = caminho_arquivo.name

    # Extração do ticker a partir do padrão de nome Economatica
    # Ex.: 'Economatica-correcao_dividendos-AGRO3.xlsx' -> 'AGRO3'
    ticker = (
        nome_arquivo
        .replace('Economatica-correcao_dividendos-', '')
        .replace('.xlsx', '')
        .strip()
    )

    try:
        # Leitura defensiva por NOME de coluna (não posicional)
        df_individual = pd.read_excel(
            caminho_arquivo,
            skiprows=3,
            usecols=['Data', 'Fechamento'],
            na_values=['-'],
            keep_default_na=True
        )

        # Coerção explícita de tipos
        df_individual['Data'] = pd.to_datetime(df_individual['Data'], errors='coerce')
        df_individual['Fechamento'] = pd.to_numeric(df_individual['Fechamento'], errors='coerce')

        # Remoção de registros com chave temporal inválida
        df_individual = df_individual.dropna(subset=['Data'])

        # Renomeação para o ticker e definição de índice cronológico
        df_individual = df_individual.rename(columns={'Fechamento': ticker})
        df_individual = df_individual.set_index('Data')

        lista_dataframes.append(df_individual)

    except Exception as e:
        falhas_leitura.append((nome_arquivo, str(e)))
        logging.error(f"Falha na leitura de {nome_arquivo}: {e}")

print(f"[+] Ativos convertidos com sucesso: {len(lista_dataframes)}")
print(f"[+] Falhas de leitura: {len(falhas_leitura)}")

[+] Relatórios identificados (excluindo lockfiles): 496
[+] Ativos convertidos com sucesso: 496
[+] Falhas de leitura: 0


## Etapa II — Concatenação Lateral e Filtro Temporal

Junção dimensional de todas as séries individuais em uma matriz $T \times N$, onde $T$ é o número de datas únicas e $N$ é o número de ativos. O argumento `sort=False` é especificado explicitamente para suprimir o `FutureWarning` do pandas e fixar comportamento determinístico (a ordenação cronológica é feita imediatamente em seguida com `sort_index`).

O filtro temporal restringe a matriz à janela amostral declarada da pesquisa: 01/01/2010 a 31/12/2025.

In [21]:
print("[+] Iniciando concatenação lateral das séries individuais...")

# Junção por índice cronológico (sort=False explícito para evitar deprecation warning)
df_consolidado = pd.concat(lista_dataframes, axis=1, sort=False)

# Ordenação cronológica crescente do índice
df_consolidado = df_consolidado.sort_index()

# Recorte para a janela amostral declarada
df_recortado = df_consolidado.loc[DATA_INICIO:DATA_FIM].copy()

print(f"[Diagnóstico pós-concatenação]:")
print(f"      -> Matriz consolidada: {df_consolidado.shape}")
print(f"      -> Após recorte temporal [{DATA_INICIO.date()} – {DATA_FIM.date()}]: {df_recortado.shape}")
print(f"      -> Total de NaN na matriz recortada: {df_recortado.isna().sum().sum():,}")
print(f"      -> Percentual de NaN: {df_recortado.isna().sum().sum() / df_recortado.size:.2%}")

[+] Iniciando concatenação lateral das séries individuais...
[Diagnóstico pós-concatenação]:
      -> Matriz consolidada: (6849, 496)
      -> Após recorte temporal [2010-01-01 – 2025-12-31]: (4174, 496)
      -> Total de NaN na matriz recortada: 1,038,267
      -> Percentual de NaN: 50.15%


## Etapa III — Remoção de Feriados Nacionais

Eliminação das linhas em que **todos** os ativos apresentam `NaN` simultaneamente. Essas datas correspondem exclusivamente a feriados nacionais brasileiros nos quais a B3 não operou (Confraternização Universal, Carnaval, Sexta-feira Santa, Tiradentes, Corpus Christi, Independência, Finados, Proclamação da República, Consciência Negra, Natal, etc.). Os relatórios de fechamento exportados pela Economatica listam essas datas com o marcador `-`, convertido para `NaN` na Etapa I.

> **Nota metodológica:** o método `dropna(how='all')` não remove finais de semana porque a Economatica não os inclui no relatório de cotações na origem. O calendário de pregões aqui obtido é, portanto, o resultado *empírico* da intersecção dos calendários de todos os ativos do universo, e não um calendário oficial pré-carregado da B3.

In [22]:
# Remoção apenas de linhas integralmente nulas (feriados nacionais)
df_calendario = df_recortado.dropna(how='all')

linhas_removidas = df_recortado.shape[0] - df_calendario.shape[0]
total_pregoes = df_calendario.shape[0]

print(f"[Ajuste de Calendário]:")
print(f"      -> Feriados nacionais removidos: {linhas_removidas} datas")
print(f"      -> Pregões efetivos no período: {total_pregoes}")

# Verificação diagnóstica: nenhuma das datas removidas pode ser fim de semana
datas_removidas = df_recortado.index.difference(df_calendario.index)
fds_removidos = (datas_removidas.dayofweek >= 5).sum()
print(f"      -> Sanity check (deve ser 0): finais de semana entre as datas removidas = {fds_removidos}")

[Ajuste de Calendário]:
      -> Feriados nacionais removidos: 207 datas
      -> Pregões efetivos no período: 3967
      -> Sanity check (deve ser 0): finais de semana entre as datas removidas = 0


## Etapa IV — Filtro de Liquidez por Presença em Pregão

Cálculo da proporção individual de presença em pregão de cada ativo, baseado no denominador de **dias efetivos de mercado** (pós-Etapa III), e retenção apenas dos ativos com presença $\geq 95\%$.

$$P_i = \frac{\sum_{t=1}^{T} \mathbb{1}\{P_{i,t} \neq \text{NaN}\}}{T}$$

Onde $P_i$ é a presença percentual do ativo $i$, $T$ é o total de pregões efetivos da janela amostral e $\mathbb{1}\{\cdot\}$ é a função indicadora.

> **Justificativa do limiar de 95%:** o valor é equivalente ao critério de presença em pregão utilizado pela B3 como filtro de elegibilidade para inclusão na carteira teórica do índice IBOVESPA. Adotá-lo aqui alinha o universo amostral da pesquisa com o conceito de liquidez utilizado pelo próprio mercado, ao mesmo tempo em que limita a posterior imputação por *forward fill* a no máximo $5\%$ das observações por ativo, preservando a fidedignidade das estimativas de volatilidade e covariância. *Verificar a redação atual do regulamento do IBOVESPA antes da defesa para confirmar a referência.*

In [23]:
# Proporção de presença em pregão por ativo
proporcao_presenca = df_calendario.notna().sum() / total_pregoes

ativos_aprovados_liquidez = proporcao_presenca[proporcao_presenca >= LIMIAR_PRESENCA].index.tolist()
ativos_reprovados_liquidez = proporcao_presenca[proporcao_presenca < LIMIAR_PRESENCA]

df_liquidos = df_calendario[ativos_aprovados_liquidez].copy()

print(f"[Filtro de Liquidez ({LIMIAR_PRESENCA:.0%} de presença)]:")
print(f"      -> Ativos aprovados:   {len(ativos_aprovados_liquidez)}")
print(f"      -> Ativos reprovados:  {len(ativos_reprovados_liquidez)}")
print(f"      -> Matriz pós-filtro:  {df_liquidos.shape}")

[Filtro de Liquidez (95% de presença)]:
      -> Ativos aprovados:   135
      -> Ativos reprovados:  361
      -> Matriz pós-filtro:  (3967, 135)


## Etapa V — Filtro de Integridade Inicial (Mitigação de Lookahead Bias)

Exclusão dos ativos cujo **primeiro pregão real** (primeira data com cotação não-nula) é posterior ao início da janela amostral. Esta etapa, inexistente no pipeline anterior, corrige um defeito metodológico relevante:

> No pipeline original, a imputação combinada `ffill().bfill()` era aplicada após o filtro de liquidez. Como o `bfill` propaga valores **para trás** no tempo, ele preenchia retroativamente datas anteriores ao próprio IPO de um ativo com o preço de seu primeiro pregão. Para um ativo cujo primeiro pregão ocorreu em abril de 2010 (admitido pelo filtro de 95%), o `bfill` gerava ~70 observações fictícias com preço constante entre janeiro e abril de 2010 — caracterizando *lookahead bias* explícito e produzindo variância e covariâncias artificialmente subestimadas para a fase pré-IPO.

A exclusão por integridade inicial é a alternativa metodologicamente mais rigorosa: o universo amostral final contém apenas ativos com histórico **íntegro desde a abertura da janela**, o que viabiliza o uso exclusivo de `ffill` para lacunas residuais legítimas (suspensões, leilões intradiários) sem qualquer contaminação retroativa.

> **Implicação para o universo amostral:** o filtro intensifica o viés de sobrevivência (*survivorship bias*) já presente no filtro de 95%. O universo final é, por construção, composto de empresas com listagem ativa e contínua desde o início da janela amostral — empresas grandes, estabelecidas e líquidas. IPOs posteriores a 04/01/2010 são automaticamente excluídos. Esta restrição **deve ser explicitada na seção de limitações do TCC**: as conclusões aplicam-se ao subconjunto de blue chips e *mid caps* maduras da B3, e não ao universo total de ações listadas no período.

In [24]:
# Identificação do primeiro pregão real (primeira data com cotação não-nula) por ativo
data_inicio_janela = df_liquidos.index.min()
primeiro_pregao_real = df_liquidos.apply(lambda s: s.dropna().index.min())

# Critério de integridade: o ativo deve ter dados desde o início da janela
ativos_integros = primeiro_pregao_real[primeiro_pregao_real <= data_inicio_janela].index.tolist()
ativos_ipo_tardio = primeiro_pregao_real[primeiro_pregao_real > data_inicio_janela]

df_integros = df_liquidos[ativos_integros].copy()

print(f"[Filtro de Integridade Inicial]:")
print(f"      -> Data de início efetiva da janela: {data_inicio_janela.date()}")
print(f"      -> Ativos com histórico íntegro: {len(ativos_integros)}")
print(f"      -> Ativos excluídos por IPO tardio: {len(ativos_ipo_tardio)}")
if len(ativos_ipo_tardio) > 0:
    print(f"\n      Detalhe dos excluídos:")
    for ticker, data in ativos_ipo_tardio.sort_values().items():
        gap_dias = (data - data_inicio_janela).days
        print(f"         - {ticker}: primeiro pregão em {data.date()} ({gap_dias} dias após o início)")

[Filtro de Integridade Inicial]:
      -> Data de início efetiva da janela: 2010-01-04
      -> Ativos com histórico íntegro: 131
      -> Ativos excluídos por IPO tardio: 4

      Detalhe dos excluídos:
         - CGRA4: primeiro pregão em 2010-01-07 (3 dias após o início)
         - ECOR3: primeiro pregão em 2010-03-31 (86 dias após o início)
         - MILS3: primeiro pregão em 2010-04-15 (101 dias após o início)
         - SIMH3: primeiro pregão em 2010-04-20 (106 dias após o início)


## Etapa VI — Imputação Residual por Forward Fill

Aplicação **exclusiva** de *Forward Fill* ($\text{ffill}$) sobre lacunas residuais legítimas — dias em que um papel específico não negociou apesar de a B3 estar aberta (suspensão temporária, retenção em leilão de volatilidade, ausência circunstancial de contraparte).

A escolha pelo *ffill* exclusivo baseia-se em três argumentos:

1. **Aderência à microestrutura de mercado:** quando um ativo não negocia em um pregão, sua última cotação observada permanece como a melhor estimativa pública de seu valor de fechamento.
2. **Ausência de *lookahead bias*:** o operador $\text{ffill}$ utiliza estritamente informação disponível em $t-k$ para preencher $t$, com $k \geq 1$. Não há contaminação por dados futuros.
3. **Compatibilidade com o filtro da Etapa V:** como o universo amostral final contém apenas ativos com primeiro pregão real $\leq$ início da janela, não há `NaN` na primeira linha da matriz, dispensando qualquer necessidade de *backward fill*.

In [25]:
# Imputação por propagação prospectiva (forward fill exclusivo)
df_sanitizado = df_integros.ffill()

nulos_residuais = df_sanitizado.isna().sum().sum()
print(f"[Imputação Residual]:")
print(f"      -> Lacunas preenchidas por ffill: {df_integros.isna().sum().sum() - nulos_residuais}")
print(f"      -> NaN remanescentes: {nulos_residuais}")
print(f"      -> Matriz final: {df_sanitizado.shape}")

[Imputação Residual]:
      -> Lacunas preenchidas por ffill: 1850
      -> NaN remanescentes: 0
      -> Matriz final: (3967, 131)


## Etapa VII — Testes de Integridade Pós-Processamento

Bateria de verificações sobre a matriz final antes da persistência. Qualquer falha aqui interrompe o pipeline via `AssertionError`, evitando que dados corrompidos sejam propagados para as etapas seguintes (cálculo de retornos, otimização de portfólios).

In [26]:
print("[+] Executando testes de integridade pós-processamento...")

# T1: ausência de NaN
assert df_sanitizado.isna().sum().sum() == 0, "Falha T1: há NaN residual na matriz final"
print("      [OK] T1 — Ausência de NaN")

# T2: índice sem duplicatas
assert not df_sanitizado.index.has_duplicates, "Falha T2: índice cronológico contém duplicatas"
print("      [OK] T2 — Índice sem duplicatas")

# T3: índice monotônico crescente
assert df_sanitizado.index.is_monotonic_increasing, "Falha T3: índice não está em ordem cronológica crescente"
print("      [OK] T3 — Índice monotônico crescente")

# T4: todos os preços estritamente positivos
assert (df_sanitizado > 0).all().all(), "Falha T4: há preços não-positivos na matriz"
print("      [OK] T4 — Todos os preços > 0")

# T5: todas as colunas com presença = 100% (consequência das Etapas IV e V)
presenca_final = df_sanitizado.notna().sum() / len(df_sanitizado)
assert (presenca_final == 1.0).all(), "Falha T5: há ativos com presença < 100% pós-imputação"
print("      [OK] T5 — Presença em pregão = 100% para todos os ativos")

print("\n[OK] Matriz sanitizada aprovada em todos os testes de integridade.")

[+] Executando testes de integridade pós-processamento...
      [OK] T1 — Ausência de NaN
      [OK] T2 — Índice sem duplicatas
      [OK] T3 — Índice monotônico crescente
      [OK] T4 — Todos os preços > 0
      [OK] T5 — Presença em pregão = 100% para todos os ativos

[OK] Matriz sanitizada aprovada em todos os testes de integridade.


## Etapa VIII — Persistência da Matriz Final e Log Auditável

Gravação da matriz sanitizada e, em paralelo, de um log auditável dos ativos descartados em cada etapa de filtragem. O log de exclusões é exigência de transparência metodológica: permite à banca verificar quais ativos foram removidos e por que motivo, sem necessidade de re-executar o pipeline.

In [27]:
# Caminhos de saída
caminho_matriz = DIRETORIO_DESTINO / "Matriz_precos_sanitizada.csv"
caminho_log    = DIRETORIO_DESTINO / "tickers_excluidos.csv"

# 1) Persistência da matriz sanitizada
df_sanitizado.to_csv(caminho_matriz, index=True)

# 2) Construção do log auditável de tickers excluídos
log_excluidos = []

# Excluídos pelo filtro de liquidez (Etapa IV)
for ticker, presenca in ativos_reprovados_liquidez.items():
    log_excluidos.append({
        'ticker': ticker,
        'presenca_pct': round(presenca * 100, 2),
        'primeiro_pregao_real': df_calendario[ticker].dropna().index.min() if df_calendario[ticker].notna().any() else None,
        'motivo': 'Presenca em pregao < 95% (Etapa IV)'
    })

# Excluídos pelo filtro de integridade inicial (Etapa V)
for ticker, data_inicio_ticker in ativos_ipo_tardio.items():
    log_excluidos.append({
        'ticker': ticker,
        'presenca_pct': round(proporcao_presenca[ticker] * 100, 2),
        'primeiro_pregao_real': data_inicio_ticker,
        'motivo': 'Primeiro pregao real posterior ao inicio da janela (Etapa V)'
    })

# Adiciona também as falhas de leitura, se houver
for nome_arq, msg_erro in falhas_leitura:
    log_excluidos.append({
        'ticker': nome_arq,
        'presenca_pct': None,
        'primeiro_pregao_real': None,
        'motivo': f'Falha de leitura na ingestao: {msg_erro[:80]}'
    })

df_log = pd.DataFrame(log_excluidos).sort_values(['motivo', 'ticker'])
df_log.to_csv(caminho_log, index=False)

# Relatório final
print("=" * 70)
print("PIPELINE EXECUTADO COM SUCESSO")
print("=" * 70)
print(f"Matriz final:  {df_sanitizado.shape[0]} pregoes x {df_sanitizado.shape[1]} ativos")
print(f"               Periodo: {df_sanitizado.index.min().date()} a {df_sanitizado.index.max().date()}")
print(f"Saida:         {caminho_matriz}")
print(f"Log:           {caminho_log}  ({len(df_log)} ativos descartados)")
print("=" * 70)

PIPELINE EXECUTADO COM SUCESSO
Matriz final:  3967 pregoes x 131 ativos
               Periodo: 2010-01-04 a 2025-12-30
Saida:         c:\VSCodeWorkspace\TCC_Final\data\Matriz_Precos\Matriz_precos_sanitizada.csv
Log:           c:\VSCodeWorkspace\TCC_Final\data\Matriz_Precos\tickers_excluidos.csv  (365 ativos descartados)


## Texto-Prosa para Citação Direta no TCC

> *"O tratamento das séries históricas de fechamento foi conduzido em pipeline único, integrando ingestão, alinhamento temporal, filtragem por liquidez e sanitização contra vieses de imputação. Após a consolidação dos relatórios brutos da Economatica em uma matriz $T \times N$ e a aplicação do recorte temporal entre 01/01/2010 e 31/12/2025, foram eliminadas as datas em que a B3 não operou (feriados nacionais), obtendo-se $T = 3967$ pregões efetivos. Sobre essa base, aplicou-se um critério de liquidez por presença em pregão de no mínimo 95%, alinhado ao requisito de elegibilidade utilizado pela B3 para a carteira teórica do índice IBOVESPA. Para mitigar o viés de retrospecção (lookahead bias) que seria introduzido pela imputação retroativa de cotações pré-IPO via backward fill, adotou-se filtro adicional de integridade inicial: foram excluídos os ativos cujo primeiro pregão real foi posterior à data de abertura da janela amostral. Lacunas residuais (suspensões e retenções em leilão de volatilidade) foram preenchidas exclusivamente por forward fill, operador que utiliza apenas informação observada em $t-k$, com $k \geq 1$. O universo amostral final é constituído pelos ativos que satisfizeram simultaneamente os três critérios — liquidez, integridade temporal e leitura bem-sucedida — reconhecendo-se que a exigência de continuidade desde 2010 impõe um viés de sobrevivência inerente, restringindo as conclusões da pesquisa ao subconjunto de empresas com listagem ativa e líquida ao longo de todo o período analisado."*

================================================================================
**APÊNDICE A – PIPELINE INTEGRADO DE TRATAMENTO E SANITIZAÇÃO DE DADOS**
================================================================================

Este apêndice documenta, em forma integral, o algoritmo computacional desenvolvido em linguagem Python para automatizar a etapa de extração, alinhamento cronológico, filtragem por liquidez e sanitização das séries históricas de preços dos ativos negociados na B3. Os dados brutos individuais foram extraídos da plataforma Economatica em formato `.xlsx`, contendo cotações de fechamento ajustadas por proventos.

A rotina foi projetada para preservar o rigor metodológico da pesquisa, operando sob as seguintes premissas técnicas e restrições analíticas:

1. **Robustez da ingestão:** o algoritmo descarta automaticamente os *lockfiles* temporários do Microsoft Excel (arquivos com prefixo `~$`) e realiza a seleção de colunas por nome (`Data`, `Fechamento`), garantindo resistência a eventuais reorganizações de schema na exportação da Economatica.

2. **Tratamento explícito do marcador de feriado:** o caractere `-`, utilizado pela Economatica para sinalizar dias sem pregão, é convertido em `NaN` na ingestão, permitindo o tratamento subsequente como ausência genuína de observação.

3. **Sincronização cronológica:** os DataFrames individuais são unidos por concatenação lateral indexada por data, com ordenação cronológica explícita do índice resultante.

4. **Filtragem temporal:** aplicação de recorte estrito da janela amostral entre 01/01/2010 e 31/12/2025.

5. **Identificação empírica do calendário de pregões:** remoção das linhas em que todos os ativos apresentam `NaN` concomitante, correspondendo aos feriados nacionais brasileiros em que a B3 não operou.

6. **Filtro de liquidez por presença em pregão:** retenção de ativos com presença $\geq 95\%$ sobre os pregões efetivos, em alinhamento com o critério de elegibilidade do índice IBOVESPA.

7. **Filtro de integridade inicial:** exclusão de ativos cujo primeiro pregão real é posterior à data de abertura da janela amostral, eliminando a necessidade de imputação retroativa e, com ela, o risco de contaminação por *lookahead bias*.

8. **Imputação residual conservadora:** preenchimento de lacunas remanescentes exclusivamente por *Forward Fill*, sem aplicação de *Backward Fill*, preservando a propriedade de causalidade temporal do operador de imputação.

9. **Testes de integridade pós-processamento:** verificação programática de ausência de `NaN`, unicidade do índice cronológico, monotonicidade temporal, positividade dos preços e presença plena de cada ativo.

10. **Persistência auditável:** gravação da matriz final em formato CSV e registro paralelo, em arquivo separado, dos ativos excluídos em cada etapa de filtragem, com indicação explícita do motivo de exclusão.

O código-fonte integral, estruturado para execução em células sequenciais de cadernos interativos Jupyter, encontra-se transcrito na íntegra a seguir:

[INSERIR O CÓDIGO DAS CÉLULAS DE EXECUÇÃO DESTE NOTEBOOK NESTE ESPAÇO — sugestão: exportar via `jupyter nbconvert --to script` e colar o conteúdo das células `code` aqui, ou copiar manualmente das células 3, 5, 7, 9, 11, 13, 15, 17 e 19.]


from pathlib import Path
import logging
import pandas as pd
import os

# Configuração do sistema de logs
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] - %(message)s"
)

# Caminhos relativos à raiz do projeto (usando pathlib puramente)
parent_path = Path.cwd().parent.parent

# Substitua o os.path.join por operadores de barra (/) do pathlib
DIRETORIO_ORIGEM = parent_path / "data" / "dados_economatica"
DIRETORIO_DESTINO = parent_path / "data" / "Matriz_Precos"

# Agora você pode descomentar e usar o método do pathlib com segurança:
DIRETORIO_DESTINO.mkdir(parents=True, exist_ok=True)

# Criação automatizada do diretório de destino
#DIRETORIO_DESTINO.mkdir(parents=True, exist_ok=True)

# Parâmetros metodológicos do pipeline (concentrados para auditoria)
DATA_INICIO = pd.to_datetime('2010-01-01')
DATA_FIM    = pd.to_datetime('2025-12-31')
LIMIAR_PRESENCA = 0.95   # Critério de presença em pregão (alinhado ao IBOVESPA)

print(f"[OK] Diretório de origem:  {DIRETORIO_ORIGEM}")
print(f"[OK] Diretório de destino: {DIRETORIO_DESTINO}")
print(f"[OK] Janela temporal: {DATA_INICIO.date()} a {DATA_FIM.date()}")
print(f"[OK] Limiar de presença:  {LIMIAR_PRESENCA:.0%}")

# Mapeamento dos arquivos Excel, com exclusão de lockfiles temporários do Office
arquivos_excel = [
    p for p in DIRETORIO_ORIGEM.glob("*.xlsx")
    if not p.name.startswith("~$")
]
print(f"[+] Relatórios identificados (excluindo lockfiles): {len(arquivos_excel)}")

# Lista para armazenamento dos DataFrames individuais preparados
lista_dataframes = []
falhas_leitura = []

for caminho_arquivo in arquivos_excel:
    nome_arquivo = caminho_arquivo.name

    # Extração do ticker a partir do padrão de nome Economatica
    # Ex.: 'Economatica-correcao_dividendos-AGRO3.xlsx' -> 'AGRO3'
    ticker = (
        nome_arquivo
        .replace('Economatica-correcao_dividendos-', '')
        .replace('.xlsx', '')
        .strip()
    )

    try:
        # Leitura defensiva por NOME de coluna (não posicional)
        df_individual = pd.read_excel(
            caminho_arquivo,
            skiprows=3,
            usecols=['Data', 'Fechamento'],
            na_values=['-'],
            keep_default_na=True
        )

        # Coerção explícita de tipos
        df_individual['Data'] = pd.to_datetime(df_individual['Data'], errors='coerce')
        df_individual['Fechamento'] = pd.to_numeric(df_individual['Fechamento'], errors='coerce')

        # Remoção de registros com chave temporal inválida
        df_individual = df_individual.dropna(subset=['Data'])

        # Renomeação para o ticker e definição de índice cronológico
        df_individual = df_individual.rename(columns={'Fechamento': ticker})
        df_individual = df_individual.set_index('Data')

        lista_dataframes.append(df_individual)

    except Exception as e:
        falhas_leitura.append((nome_arquivo, str(e)))
        logging.error(f"Falha na leitura de {nome_arquivo}: {e}")

print(f"[+] Ativos convertidos com sucesso: {len(lista_dataframes)}")
print(f"[+] Falhas de leitura: {len(falhas_leitura)}")

print("[+] Iniciando concatenação lateral das séries individuais...")

# Junção por índice cronológico (sort=False explícito para evitar deprecation warning)
df_consolidado = pd.concat(lista_dataframes, axis=1, sort=False)

# Ordenação cronológica crescente do índice
df_consolidado = df_consolidado.sort_index()

# Recorte para a janela amostral declarada
df_recortado = df_consolidado.loc[DATA_INICIO:DATA_FIM].copy()

print(f"[Diagnóstico pós-concatenação]:")
print(f"      -> Matriz consolidada: {df_consolidado.shape}")
print(f"      -> Após recorte temporal [{DATA_INICIO.date()} – {DATA_FIM.date()}]: {df_recortado.shape}")
print(f"      -> Total de NaN na matriz recortada: {df_recortado.isna().sum().sum():,}")
print(f"      -> Percentual de NaN: {df_recortado.isna().sum().sum() / df_recortado.size:.2%}")

# Remoção apenas de linhas integralmente nulas (feriados nacionais)
df_calendario = df_recortado.dropna(how='all')

linhas_removidas = df_recortado.shape[0] - df_calendario.shape[0]
total_pregoes = df_calendario.shape[0]

print(f"[Ajuste de Calendário]:")
print(f"      -> Feriados nacionais removidos: {linhas_removidas} datas")
print(f"      -> Pregões efetivos no período: {total_pregoes}")

# Verificação diagnóstica: nenhuma das datas removidas pode ser fim de semana
datas_removidas = df_recortado.index.difference(df_calendario.index)
fds_removidos = (datas_removidas.dayofweek >= 5).sum()
print(f"      -> Sanity check (deve ser 0): finais de semana entre as datas removidas = {fds_removidos}")

# Proporção de presença em pregão por ativo
proporcao_presenca = df_calendario.notna().sum() / total_pregoes

ativos_aprovados_liquidez = proporcao_presenca[proporcao_presenca >= LIMIAR_PRESENCA].index.tolist()
ativos_reprovados_liquidez = proporcao_presenca[proporcao_presenca < LIMIAR_PRESENCA]

df_liquidos = df_calendario[ativos_aprovados_liquidez].copy()

print(f"[Filtro de Liquidez ({LIMIAR_PRESENCA:.0%} de presença)]:")
print(f"      -> Ativos aprovados:   {len(ativos_aprovados_liquidez)}")
print(f"      -> Ativos reprovados:  {len(ativos_reprovados_liquidez)}")
print(f"      -> Matriz pós-filtro:  {df_liquidos.shape}")

# Identificação do primeiro pregão real (primeira data com cotação não-nula) por ativo
data_inicio_janela = df_liquidos.index.min()
primeiro_pregao_real = df_liquidos.apply(lambda s: s.dropna().index.min())

# Critério de integridade: o ativo deve ter dados desde o início da janela
ativos_integros = primeiro_pregao_real[primeiro_pregao_real <= data_inicio_janela].index.tolist()
ativos_ipo_tardio = primeiro_pregao_real[primeiro_pregao_real > data_inicio_janela]

df_integros = df_liquidos[ativos_integros].copy()

print(f"[Filtro de Integridade Inicial]:")
print(f"      -> Data de início efetiva da janela: {data_inicio_janela.date()}")
print(f"      -> Ativos com histórico íntegro: {len(ativos_integros)}")
print(f"      -> Ativos excluídos por IPO tardio: {len(ativos_ipo_tardio)}")
if len(ativos_ipo_tardio) > 0:
    print(f"\n      Detalhe dos excluídos:")
    for ticker, data in ativos_ipo_tardio.sort_values().items():
        gap_dias = (data - data_inicio_janela).days
        print(f"         - {ticker}: primeiro pregão em {data.date()} ({gap_dias} dias após o início)")

# Imputação por propagação prospectiva (forward fill exclusivo)
df_sanitizado = df_integros.ffill()

nulos_residuais = df_sanitizado.isna().sum().sum()
print(f"[Imputação Residual]:")
print(f"      -> Lacunas preenchidas por ffill: {df_integros.isna().sum().sum() - nulos_residuais}")
print(f"      -> NaN remanescentes: {nulos_residuais}")
print(f"      -> Matriz final: {df_sanitizado.shape}")

print("[+] Executando testes de integridade pós-processamento...")

# T1: ausência de NaN
assert df_sanitizado.isna().sum().sum() == 0, "Falha T1: há NaN residual na matriz final"
print("      [OK] T1 — Ausência de NaN")

# T2: índice sem duplicatas
assert not df_sanitizado.index.has_duplicates, "Falha T2: índice cronológico contém duplicatas"
print("      [OK] T2 — Índice sem duplicatas")

# T3: índice monotônico crescente
assert df_sanitizado.index.is_monotonic_increasing, "Falha T3: índice não está em ordem cronológica crescente"
print("      [OK] T3 — Índice monotônico crescente")

# T4: todos os preços estritamente positivos
assert (df_sanitizado > 0).all().all(), "Falha T4: há preços não-positivos na matriz"
print("      [OK] T4 — Todos os preços > 0")

# T5: todas as colunas com presença = 100% (consequência das Etapas IV e V)
presenca_final = df_sanitizado.notna().sum() / len(df_sanitizado)
assert (presenca_final == 1.0).all(), "Falha T5: há ativos com presença < 100% pós-imputação"
print("      [OK] T5 — Presença em pregão = 100% para todos os ativos")

print("\n[OK] Matriz sanitizada aprovada em todos os testes de integridade.")

# Caminhos de saída
caminho_matriz = DIRETORIO_DESTINO / "Matriz_precos_sanitizada.csv"
caminho_log    = DIRETORIO_DESTINO / "tickers_excluidos.csv"

# 1) Persistência da matriz sanitizada
df_sanitizado.to_csv(caminho_matriz, index=True)

# 2) Construção do log auditável de tickers excluídos
log_excluidos = []

# Excluídos pelo filtro de liquidez (Etapa IV)
for ticker, presenca in ativos_reprovados_liquidez.items():
    log_excluidos.append({
        'ticker': ticker,
        'presenca_pct': round(presenca * 100, 2),
        'primeiro_pregao_real': df_calendario[ticker].dropna().index.min() if df_calendario[ticker].notna().any() else None,
        'motivo': 'Presenca em pregao < 95% (Etapa IV)'
    })

# Excluídos pelo filtro de integridade inicial (Etapa V)
for ticker, data_inicio_ticker in ativos_ipo_tardio.items():
    log_excluidos.append({
        'ticker': ticker,
        'presenca_pct': round(proporcao_presenca[ticker] * 100, 2),
        'primeiro_pregao_real': data_inicio_ticker,
        'motivo': 'Primeiro pregao real posterior ao inicio da janela (Etapa V)'
    })

# Adiciona também as falhas de leitura, se houver
for nome_arq, msg_erro in falhas_leitura:
    log_excluidos.append({
        'ticker': nome_arq,
        'presenca_pct': None,
        'primeiro_pregao_real': None,
        'motivo': f'Falha de leitura na ingestao: {msg_erro[:80]}'
    })

df_log = pd.DataFrame(log_excluidos).sort_values(['motivo', 'ticker'])
df_log.to_csv(caminho_log, index=False)

# Relatório final
print("=" * 70)
print("PIPELINE EXECUTADO COM SUCESSO")
print("=" * 70)
print(f"Matriz final:  {df_sanitizado.shape[0]} pregoes x {df_sanitizado.shape[1]} ativos")
print(f"               Periodo: {df_sanitizado.index.min().date()} a {df_sanitizado.index.max().date()}")
print(f"Saida:         {caminho_matriz}")
print(f"Log:           {caminho_log}  ({len(df_log)} ativos descartados)")
print("=" * 70)